# Notebook 02 — Indexing & Slicing

**numpy-mastery** · Module 02 of 06

> **Goal**: select, extract, and modify any part of an array — with basic indexing,
> slicing, fancy indexing, and boolean masking.

**What you'll be able to do after this notebook:**
- Extract any element, row, column, or sub-matrix with one expression
- Understand the difference between a **view** and a **copy** — and why it matters
- Select elements by condition using boolean masks
- Use `np.where` as a vectorised if/else

---


In [1]:
import numpy as np

---
## 1 · Basic indexing — 1-D

Indexing works exactly like Python lists, with one bonus: **negative indices**
count from the end.

In [6]:
arr = np.array([10, 20, 30, 40, 50])
#               0    1   2   3   4   ← positive indices
#              -5   -4  -3  -2  -1   ← negative indices

print(arr[0])
print(arr[-1])
print(arr[-2])

#Assignment works the same way 
arr[2] = 99
print(arr)

10
50
40
[10 20 99 40 50]


---
## 2 · Slicing — 1-D

Syntax: `arr[start : stop : step]`

| Part | Default | Meaning |
|------|---------|---------|
| `start` | `0` | index to begin (inclusive) |
| `stop` | `len` | index to end (**exclusive**) |
| `step` | `1` | jump between elements |

In [26]:
arr = np.array([10, 20, 30, 40, 50, 60, 70, 80])
#                0   1   2   3   4   5   6   7  
#               -8  -7  -6  -5  -4  -3  -2  -1

print(arr[2:6])     # [30 40 50 60]  — indices 2,3,4,5
print(arr[-6:-2])   # [30 40 50 60]  — indices -6,-5,-4,-3
print(arr[:4])      # [10 20 30 40]  — from start to 3
print(arr[4:])      # [50 60 70 80]  — from 4 to end
print(arr[::2])     # [10 30 50 70]  — every other element
print(arr[1::2])    # [20 40 60 80]  — every other, starting at 1
print(arr[::-1])    # [80 70 60 50 40 30 20 10]  — reversed
print(arr[6:1:-2])  # [70 50 30]  — step=-2, going backwards


# The slicing always goes from left to right
# if you try to reverse the order withou going backward you will have an empty array
print(25 * "-")
print(arr[6:2])
print(arr[-2:-6])



[30 40 50 60]
[30 40 50 60]
[10 20 30 40]
[50 60 70 80]
[10 30 50 70]
[20 40 60 80]
[80 70 60 50 40 30 20 10]
[70 50 30]
-------------------------
[]
[]


### Slices return VIEWS, not copies

A view shares memory with the original array — modifying it **modifies the original**.

In [30]:
original = np.array([1, 2, 3, 4, 5])
view = original[1:4]    # [2, 3, 4]

view[0] = 99 
print(original)     # [1, 99, 3, 4, 5]  ← original changed!
print(25 * "-")

#To avoid this use copy 
original_1 = np.array([1, 2, 3, 4, 5])
safe_copy = original_1[1:4].copy()

safe_copy[0] = 99
print(original_1)
print(25 * "-")

# Check if two arrays share the same memory space
print(np.shares_memory(original , view))
print(np.shares_memory(original_1 , safe_copy))


[ 1 99  3  4  5]
-------------------------
[1 2 3 4 5]
-------------------------
True
False


---
## 3 · Indexing 2-D arrays

For 2-D arrays, use `arr[row, col]` — comma syntax is **always preferred**
over `arr[row][col]` (which creates an intermediate array).

In [34]:
#               0   1   2   3
M = np.array([[ 1,  2,  3,  4],     # 0 | -3
              [ 5,  6,  7,  8],     # 1 | -2
              [ 9, 10, 11, 12]])    # 2 | -1
#              -4  -3  -2  -1
print(M[0, 0])
print(M[0, 3])
print(M[1, 2])
print(M[0, 3])
print(M[-1, -1])

1
4
7
4
12


In [ ]:
# Selecting entire rows and columns
# selecting lines
print(M[0:1]) # first line
print(15 * "-")
print(M[1:2]) # Second line
print(15 * "-")
print(M[2:])  # Thirs line
print(15 * "-")
print(M[1: ]) # last Two lines
print(15 * "-")
print(M[: 2]) # First Two lines

[[1 2 3 4]]
---------------
[[5 6 7 8]]
---------------
[[ 9 10 11 12]]
---------------
[[ 5  6  7  8]
 [ 9 10 11 12]]
---------------
[[1 2 3 4]
 [5 6 7 8]]


In [53]:
# Selecting Columns
print(M[: , 0:1]) # first line
print(15 * "-")
print(M[: , 1:2]) # Second column
print(15 * "-")
print(M[: ,2:3]) # Thrid column

[[1]
 [5]
 [9]]
---------------
[[ 2]
 [ 6]
 [10]]
---------------
[[ 3]
 [ 7]
 [11]]


In [56]:
# submatrix
print(M[0:2 , 1:3])

[[2 3]
 [6 7]]


In [57]:
# Every other row
print(M[::2 , :])

[[ 1  2  3  4]
 [ 9 10 11 12]]


In [58]:
# Reverse column Order
print(M[: , ::-1])

[[ 4  3  2  1]
 [ 8  7  6  5]
 [12 11 10  9]]


In [61]:
# Reverse line Order
print(M[::-1 , ::])

[[ 9 10 11 12]
 [ 5  6  7  8]
 [ 1  2  3  4]]


In [60]:
print(M[::-1 , ::-1])

[[12 11 10  9]
 [ 8  7  6  5]
 [ 4  3  2  1]]


### Integer index vs slice — dimension dropping

This is one of the most important rules in NumPy and a very common source of bugs:

| Syntax | What happens | Result shape |
|--------|-------------|-------------|
| `M[1, :]` | **integer** on row → drops that dimension | `(4,)` — 1-D |
| `M[1:2, :]` | **slice** on row → keeps that dimension | `(1, 4)` — 2-D |
| `M[:, 2]` | **integer** on col → drops that dimension | `(3,)` — 1-D |
| `M[:, 2:3]` | **slice** on col → keeps that dimension | `(3, 1)` — 2-D |

**Rule**: an integer index reduces `ndim` by 1. A slice never does.  
This is why `M[1, :]` gives a 1-D array but `M[1:2, :]` gives a 2-D array.


In [2]:
M = np.array([[ 1,  2,  3,  4],
              [ 5,  6,  7,  8],
              [ 9, 10, 11, 12]])

# Integer index — DROPS the dimension
row_1d = M[1, :]     # shape (4,)   <- 1-D
col_1d = M[:, 2]     # shape (3,)   <- 1-D

# Slice — KEEPS the dimension
row_2d = M[1:2, :]   # shape (1, 4) <- still 2-D
col_2d = M[:, 2:3]   # shape (3, 1) <- still 2-D

print("M[1, :]    shape:", row_1d.shape, "->", row_1d)
print("M[1:2, :]  shape:", row_2d.shape, "->", row_2d)
print()
print("M[:, 2]    shape:", col_1d.shape, "->", col_1d)
print("M[:, 2:3]  shape:", col_2d.shape)
print(col_2d)


M[1, :]    shape: (4,) -> [5 6 7 8]
M[1:2, :]  shape: (1, 4) -> [[5 6 7 8]]

M[:, 2]    shape: (3,) -> [ 3  7 11]
M[:, 2:3]  shape: (3, 1)
[[ 3]
 [ 7]
 [11]]


---
## 4 · How 2-D slices map to memory

A 2-D array is stored as a flat block in memory (row-major / C order by default).
Slicing just changes the **start pointer**, **shape**, and **strides** — no data is copied.

```
M = [[ 1  2  3  4]       Memory: [1][2][3][4][5][6][7][8][9][10][11][12]
     [ 5  6  7  8]
     [ 9 10 11 12]]

M[0:2, 1:3] selects:          [2][3]
                               [6][7]

It does this by:
  start  = &M[0,1]  (element 2)
  shape  = (2, 2)
  stride = (32, 8)  — jump 32 bytes for next row, 8 for next col (float64)
```

No data is moved. That's why slices are O(1) regardless of size.


In [ ]:
M = np.array([[ 1,  2,  3,  4],
              [ 5,  6,  7,  8],
              [ 9, 10, 11, 12]])

# Integer index — DROPS the dimension
row_1d = M[1, :]     # shape (4,)   <- 1-D
col_1d = M[:, 2]     # shape (3,)   <- 1-D

# Slice — KEEPS the dimension
row_2d = M[1:2, :]   # shape (1, 4) <- still 2-D
col_2d = M[:, 2:3]   # shape (3, 1) <- still 2-D

print("M[1, :]    shape:", row_1d.shape, "->", row_1d)
print("M[1:2, :]  shape:", row_2d.shape, "->", row_2d)
print()
print("M[:, 2]    shape:", col_1d.shape, "->", col_1d)
print("M[:, 2:3]  shape:", col_2d.shape)
print(col_2d)

---
## 5 · Fancy indexing — select with arrays of integers

Unlike slicing, fancy indexing always returns a **copy**.


In [64]:
arr = np.array([100, 200, 300, 400, 500])

# Select by list of indices
idx = np.array([0, 2, 4])
print(arr[idx])
print()

# Select in any order, including repeats
print(arr[[4, 2, 2, 3]])
print()

# verify it returns a copy
selected = arr[[0, 2 , 3]]
selected[0] = 99
print(selected)
print(arr)



[100 300 500]

[500 300 300 400]

[ 99 300 400]
[100 200 300 400 500]


In [73]:
# 2-D fansy indexing 
M = np.arange(16).reshape(4, 4)
print(M)
print()

# Select specsific row, column pair
rows = np.array([0, 2, 1])
print(M[rows])
print()
cols = np.array([0, 1, 3])
print(M[:,cols])
print()
print(M[rows, cols])      

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]

[[ 0  1  2  3]
 [ 8  9 10 11]
 [ 4  5  6  7]]

[[ 0  1  3]
 [ 4  5  7]
 [ 8  9 11]
 [12 13 15]]

[0 9 7]


In [75]:
# Select full rows by index
print(M[[0, 2]])          # rows 0 and 2 → shape (2, 4)
print()
print(M[[3, 1, 0]])       # rows in custom orde

[[ 0  1  2  3]
 [ 8  9 10 11]]

[[12 13 14 15]
 [ 4  5  6  7]
 [ 0  1  2  3]]


---
## 6 · Boolean indexing (masking)

A boolean array used as an index selects only the `True` positions.
This is one of NumPy's most powerful and frequently used features.


In [80]:
arr = np.array([3, -1, 4, -1, 5, 9, -2, 6])

# Step 1: create a boolean mask
mask = arr> 0
print(mask)

# Step 2: apply it — always returns a COPY
positive = arr[mask]
print(positive)

# Inline (most common style)
print(arr[arr > 0])
print(arr[arr % 2 == 0])  # even numbers
 

[ True False  True False  True  True False  True]
[3 4 5 9 6]
[3 4 5 9 6]
[ 4 -2  6]


In [83]:
# Combining conditions — use & (and), | (or), ~ (not)
# NOT Python's 'and'/'or' — those don't work element-wise!

arr = np.array([3, -1, 4, -1, 5, 9, -2, 6])

print(arr[(arr > 0) &(arr < 6)  ] )
print(arr[(arr > 0) |(arr < 6)  ] )
print(arr[~(arr > 0)])               # [-1 -1 -2] — NOT positive

[3 4 5]
[ 3 -1  4 -1  5  9 -2  6]
[-1 -1 -2]


In [86]:
# In-place assignment via boolean mask
arr = np.array([3, -1, 4, -1, 5, 9, -2, 6], dtype=float)

arr[arr < 0] = 0    # ReLU activation — replace negatives with 0
print(arr)


# Replace a value abouve a threshold 
arr[arr > 5 ] = -99
print(arr)

[3. 0. 4. 0. 5. 9. 0. 6.]
[  3.   0.   4.   0.   5. -99.   0. -99.]


---
## 7 · `np.where` — vectorised if/else

`np.where(condition, value_if_true, value_if_false)` applies an element-wise
conditional — much faster than a Python loop.


In [87]:
arr = np.array([3, -1, 4, -1, 5, 9, -2, 6])

# Vectorised ternary: positive → keep, negative → replace with 0
result = np.where(arr > 0, arr, 0)
print(result)     # [3 0 4 0 5 9 0 6]  — ReLU

# Replace with different values
clipped = np.where(arr > 5, 5, np.where(arr < 0, 0, arr))
print(clipped)    # [3 0 4 0 5 5 0 5]

# With strings (useful for labels)
labels = np.where(arr >= 0, 'positive', 'negative')
print(labels)

# Single-argument form — returns INDICES where condition is True
indices = np.where(arr < 0)
print(indices)    # (array([1, 3, 6]),)
print(arr[indices])  # [-1 -1 -2]

[3 0 4 0 5 9 0 6]
[3 0 4 0 5 5 0 5]
['positive' 'negative' 'positive' 'negative' 'positive' 'positive'
 'negative' 'positive']
(array([1, 3, 6]),)
[-1 -1 -2]


---
## 8 · Useful indexing helpers

These functions are built on top of indexing but come up constantly.

In [ ]:
arr = np.array([3, 1, 4, 1, 5, 9, 2, 6, 5, 3])

# argmin / argmax — INDEX of the min/max value
print("argmin:", np.argmin(arr))   # 1  (value 1)
print("argmax:", np.argmax(arr))   # 5  (value 9)

# nonzero — indices of non-zero elements
sparse = np.array([0, 3, 0, 0, 7, 0, 2])
print("nonzero:", np.nonzero(sparse))  # (array([1, 4, 6]),)

# np.clip — clamp values to a range (no indexing needed but fits here)
print("clip [2,6]:", np.clip(arr, 2, 6))  # [3 2 4 2 5 6 2 6 5 3]

# np.take — fancy index with an axis argument
M = np.arange(12).reshape(3, 4)
print("take rows [0,2]:\n", np.take(M, [0, 2], axis=0))


---
## 9 · View vs Copy — complete reference

| Operation | View or Copy? | Notes |
|-----------|--------------|-------|
| Basic slice `arr[1:4]` | **View** | shares memory |
| Step slice `arr[::2]` | **View** | shares memory |
| Single element `arr[0]` | **Scalar** | not an array |
| `arr.reshape(...)` | **View** (usually) | copy only if non-contiguous |
| `arr.T` | **View** | shares memory |
| Fancy index `arr[[0,1,2]]` | **Copy** | always |
| Boolean mask `arr[arr>0]` | **Copy** | always |
| `arr.copy()` | **Copy** | explicit |
| `arr.flatten()` | **Copy** | always |
| `arr.ravel()` | **View** (usually) | |

**Rule of thumb**: if you need to modify the extracted data without touching
the original, always call `.copy()` explicitly.


---
## 10 · Summary cheatsheet

```python
# 1-D
arr[i]          # single element
arr[i:j]        # slice from i to j-1
arr[i:j:k]      # every k-th element
arr[::-1]        # reversed

# 2-D
M[i, j]         # single element
M[i, :]         # entire row i
M[:, j]         # entire column j
M[r0:r1, c0:c1] # sub-matrix
M[::2, :]       # every other row

# Fancy
arr[[0, 2, 4]]       # select by index list
M[[0,2], :]          # select rows 0 and 2

# Boolean
arr[arr > 0]         # filter by condition
arr[(a>0) & (a<5)]   # combined condition
arr[mask] = 0        # in-place assignment

# np.where
np.where(cond, x, y) # vectorised if/else
np.where(cond)       # indices of True elements
```
